# Task 8: Daily Batch Scoring Pipeline

Builds batch next-best-action recommendations from the latest trained VW run.
Uses deterministic segment-aware candidate slates and the same customer feature logic as Task 5.


In [ ]:
from __future__ import annotations

import io
import json
import os
import re
import shutil
import subprocess
import warnings
from datetime import datetime, timezone
from pathlib import Path

import boto3
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

def get_use_case_id(default: str = 'pl-aip-uplift') -> str:
    value = os.getenv('USE_CASE_ID', default).strip()
    if not value:
        raise ValueError('USE_CASE_ID must be non-empty.')
    return value


def get_env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {'1', 'true', 'yes', 'y'}

print('Task 8 setup loaded.')

USE_CASE_ID = get_use_case_id('pl-aip-uplift')
S3_CONFIG_BUCKET = os.getenv('S3_CONFIG_BUCKET', 'aks-nvtabular-data')
S3_DATA_BUCKET = os.getenv('S3_DATA_BUCKET', 'pratham-nvtabular-poc-data')
EMBEDDING_S3_KEY = 'logical_synthetic_data/processed/vowpal_wabbit_data/MASTER_user_context_vectors.csv'
SLATE_SIZE = 60
DECISION_POLICY = 'epsilon_greedy'
EXPLORATION_EPSILON = 0.05
ALLOW_NON_PROMOTABLE_SCORING = get_env_bool('ALLOW_NON_PROMOTABLE_SCORING', False)

s3_client = boto3.client('s3')

print(f'Use case    : {USE_CASE_ID}')
print(f'Slate size  : {SLATE_SIZE} actions per customer')
print(f'Policy      : {DECISION_POLICY} (epsilon={EXPLORATION_EPSILON:.2f})')
print(f'Allow shadow: {ALLOW_NON_PROMOTABLE_SCORING}')
print(f'Run date UTC: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")}')


In [ ]:
# ?? Helper Functions ??????????????????????????????????????????????????????

def download_s3(bucket: str, key: str, local: Path) -> None:
    local.parent.mkdir(parents=True, exist_ok=True)
    s3_client.download_file(bucket, key, str(local))


def upload_s3(bucket: str, key: str, local: Path) -> None:
    s3_client.upload_file(str(local), bucket, key)


def find_vw() -> str:
    vw = shutil.which('vw')
    if vw:
        return vw
    for p in ['/usr/bin/vw', '/bin/vw', '/opt/conda/bin/vw']:
        if Path(p).exists():
            return p
    raise RuntimeError('VW binary not found.')


def parse_s3_uri(uri: str) -> tuple[str, str]:
    if not uri.startswith('s3://'):
        raise ValueError(f'Invalid S3 URI: {uri}')
    body = uri[5:]
    bucket, _, key = body.partition('/')
    if not bucket or not key:
        raise ValueError(f'Invalid S3 URI: {uri}')
    return bucket, key


def load_json_s3(bucket: str, key: str) -> dict:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return json.loads(obj['Body'].read().decode('utf-8'))


def load_latest_run_manifest(bucket: str, use_case_id: str) -> dict:
    key = f'model_artifacts/{use_case_id}/latest_run.json'
    manifest = load_json_s3(bucket, key)
    required = ['run_id', 's3_run_prefix']
    missing = [k for k in required if not manifest.get(k)]
    if missing:
        raise ValueError(f'latest_run.json missing required fields: {missing}')
    return manifest


def load_latest_eval_manifest(bucket: str, use_case_id: str) -> dict:
    key = f'model_artifacts/{use_case_id}/latest_eval.json'
    manifest = load_json_s3(bucket, key)
    required = ['run_id', 'production_promotable', 's3_scorecard']
    missing = [k for k in required if k not in manifest]
    if missing:
        raise ValueError(f'latest_eval.json missing required fields: {missing}')
    return manifest


def merge_candidate_pools(*pools):
    merged = []
    seen = set()
    for pool in pools:
        for aid in pool or []:
            aid = int(aid)
            if aid in seen:
                continue
            merged.append(aid)
            seen.add(aid)
            if len(merged) >= SLATE_SIZE:
                return merged
    return merged


print('Helper functions loaded.')


In [ ]:
# ?? 1. Load Latest Run + Data Sources ?????????????????????????????????????

latest_run = load_latest_run_manifest(S3_CONFIG_BUCKET, USE_CASE_ID)
latest_eval = load_latest_eval_manifest(S3_CONFIG_BUCKET, USE_CASE_ID)
if latest_eval.get('run_id') != latest_run.get('run_id'):
    raise ValueError(
        f"latest_eval.json run_id {latest_eval.get('run_id')} does not match latest_run.json {latest_run.get('run_id')}. "
        'Run Task 7 for the latest Task 6 model before scoring.'
    )
production_promotable = bool(latest_eval.get('production_promotable', False))
deployment_mode = 'production' if production_promotable else 'shadow'
scoring_run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
if not production_promotable and not ALLOW_NON_PROMOTABLE_SCORING:
    raise ValueError(
        'Latest model is not production_promotable. Do not run live scoring. '
        f"Blockers: {latest_eval.get('production_blockers', [])}. "
        'Set ALLOW_NON_PROMOTABLE_SCORING=1 only to generate shadow/offline recommendations.'
    )
run_bucket, run_prefix_key = parse_s3_uri(latest_run['s3_run_prefix'])
print(f"Scoring with run_id: {latest_run['run_id']}")
print(f"Run prefix        : {latest_run['s3_run_prefix']}")
print(f"Latest eval       : {latest_eval.get('s3_scorecard')}")
print(f"Deployment mode   : {deployment_mode}")
print(f"Scoring run id    : {scoring_run_id}")

obj_full = s3_client.get_object(
    Bucket=S3_DATA_BUCKET,
    Key='logical_synthetic_data/processed/raw_splits/full_df.parquet'
)
full_df = pd.read_parquet(io.BytesIO(obj_full['Body'].read()))

obj_final = s3_client.get_object(
    Bucket=S3_CONFIG_BUCKET,
    Key=f'training_data/unified_training_{USE_CASE_ID}.parquet'
)
df_final = pd.read_parquet(io.BytesIO(obj_final['Body'].read()))

obj_lib = s3_client.get_object(
    Bucket=S3_CONFIG_BUCKET,
    Key=f'action_library/action_library_{USE_CASE_ID}.parquet'
)
action_lib = pd.read_parquet(io.BytesIO(obj_lib['Body'].read()))
ACTION_DIMENSION_COLS = ['channel', 'day', 'offer', 'time']
missing_action_dims = [c for c in ACTION_DIMENSION_COLS if c not in action_lib.columns]
if missing_action_dims:
    raise ValueError(f'Action library missing required active action dimensions: {missing_action_dims}')
action_lib = (
    action_lib.sort_values('action_id')
    .drop_duplicates(ACTION_DIMENSION_COLS, keep='first')
    .reset_index(drop=True)
)

print(f'Raw rows loaded        : {len(full_df):,}')
print(f'Unified training rows  : {len(df_final):,}')
print(f'Action library rows    : {len(action_lib):,}')


In [ ]:
# ?? 2. Build Customer Features ????????????????????????????????????????????

print('Building customer feature tables...')

demo_df = full_df[
    ['master_user_id', 'income_tier', 'city_tier', 'lifecycle_stage', 'risk_level', 'current_cibil']
].drop_duplicates('master_user_id')

product_count_df = (
    full_df.loc[full_df['product_id'].notna() & (full_df['product_id'] != 'unknown'), ['master_user_id', 'product_id']]
    .drop_duplicates()
    .groupby('master_user_id')
    .size()
    .rename('product_count')
    .reset_index()
)
demo_df = demo_df.merge(product_count_df, on='master_user_id', how='left')
demo_df['cibil'] = demo_df['current_cibil'].fillna(720).astype(int)
demo_df['product_count'] = demo_df['product_count'].fillna(0).astype(int)
demo_df = demo_df.drop(columns=['current_cibil'])
demo_lookup = demo_df.set_index('master_user_id').to_dict('index')

embedding_cols = [f'v{i}' for i in range(64)]
available_embedding_cols = [c for c in embedding_cols if c in full_df.columns]
if len(available_embedding_cols) == len(embedding_cols):
    embedding_lookup = (
        full_df[['master_user_id'] + embedding_cols]
        .drop_duplicates('master_user_id')
        .set_index('master_user_id')
    )
else:
    print(f"Loading learned context vectors from s3://{S3_DATA_BUCKET}/{EMBEDDING_S3_KEY}")
    obj_embed = s3_client.get_object(Bucket=S3_DATA_BUCKET, Key=EMBEDDING_S3_KEY)
    embed_df = pd.read_csv(io.BytesIO(obj_embed['Body'].read()))
    id_candidates = ['master_user_id', 'customer_id', 'user_id']
    id_col = next((c for c in id_candidates if c in embed_df.columns), None)
    if id_col is None:
        raise ValueError(f'Embedding file missing id column. Expected one of {id_candidates}.')
    embedding_lookup = (
        embed_df[[id_col] + embedding_cols]
        .drop_duplicates(id_col)
        .rename(columns={id_col: 'master_user_id'})
        .set_index('master_user_id')
    )

active_customers = pd.DataFrame({'master_user_id': sorted(set(df_final['master_user_id']).intersection(set(embedding_lookup.index)))})
n_active = len(active_customers)
print(f'Active customers to score: {n_active:,}')
print(f'Embeddings available     : {len(embedding_lookup):,}')
print(f'Demo records available   : {len(demo_lookup):,}')


In [ ]:
# ?? 3. Build Deterministic Segment-Aware Customer Slates ?????????????????

print('Building scoring slates...')

action_feature_cols = ['channel', 'day', 'offer', 'time']
missing_action_feature_cols = [c for c in action_feature_cols if c not in action_lib.columns]
if missing_action_feature_cols:
    raise ValueError(f'Action library missing required scoring columns: {missing_action_feature_cols}')
action_features_dict = action_lib.set_index('action_id')[action_feature_cols].to_dict('index')
valid_action_ids = set(action_features_dict.keys())
df_valid = df_final[df_final['action_id'].isin(valid_action_ids)].copy()

action_counts = (
    df_valid['action_id']
    .value_counts()
    .rename_axis('action_id')
    .reset_index(name='count')
)
global_action_pool = action_counts['action_id'].head(SLATE_SIZE).astype(int).tolist()

def build_pool_map(group_cols):
    grouped = (
        df_valid.groupby(group_cols + ['action_id']).size()
        .rename('count')
        .reset_index()
        .sort_values(group_cols + ['count', 'action_id'], ascending=[True] * len(group_cols) + [False, True])
    )
    pool_map = {}
    for key, sub in grouped.groupby(group_cols, sort=False):
        if not isinstance(key, tuple):
            key = (key,)
        pool_map[key] = sub['action_id'].head(SLATE_SIZE).astype(int).tolist()
    return pool_map

channel_pool = build_pool_map(['channel'])
day_pool = build_pool_map(['day'])
time_pool = build_pool_map(['time'])
channel_time_pool = build_pool_map(['channel', 'time'])

latest_context = (
    df_valid.sort_values(['master_user_id', 'timestamp'])
    .groupby('master_user_id', as_index=False)
    .tail(1)[['master_user_id', 'channel', 'day', 'time']]
    .drop_duplicates('master_user_id')
    .set_index('master_user_id')
)

score_ready_rows = []
skipped_no_context = 0
for uid in active_customers['master_user_id']:
    if uid not in latest_context.index:
        skipped_no_context += 1
        continue
    row = latest_context.loc[uid]
    row_channel = row.get('channel', 'CH001')
    row_day = row.get('day', 'DW001')
    row_time = row.get('time', 'TM001')
    slate = merge_candidate_pools(
        channel_time_pool.get((row_channel, row_time), []),
        channel_pool.get((row_channel,), []),
        day_pool.get((row_day,), []),
        time_pool.get((row_time,), []),
        global_action_pool,
    )[:SLATE_SIZE]
    if len(slate) != SLATE_SIZE:
        raise ValueError(f'Incomplete scoring slate for user {uid}: expected {SLATE_SIZE}, got {len(slate)}')
    score_ready_rows.append({
        'master_user_id': uid,
        'seed_channel': row_channel,
        'seed_day': row_day,
        'seed_time': row_time,
        'slate': slate,
    })

scoring_customers = pd.DataFrame(score_ready_rows)
print(f'Scoring customers ready : {len(scoring_customers):,}')
print(f'Skipped (no history ctx): {skipped_no_context:,}')
print(f'Global action pool size : {len(global_action_pool)}')


In [ ]:
# ?? 4. Build VW Scoring File (No Labels) ?????????????????????????????????

print('Building VW scoring file...')

vw_blocks = []
for idx, row in scoring_customers.iterrows():
    uid = row['master_user_id']
    if uid not in embedding_lookup.index:
        raise ValueError(f'Missing embedding for scoring customer {uid}')

    emb = embedding_lookup.loc[uid].values
    emb_str = ' '.join([f'v{i}:{round(v, 6)}' for i, v in enumerate(emb)])
    demo = demo_lookup.get(uid, {})
    user_str = (
        f"inc={demo.get('income_tier', 'NA')} "
        f"city={demo.get('city_tier', 'NA')} "
        f"life={demo.get('lifecycle_stage', 'NA')} "
        f"risk={demo.get('risk_level', 'NA')} "
        f"cibil:{demo.get('cibil', 0)} "
        f"products:{demo.get('product_count', 0)}"
    )
    block_lines = [f'shared |emb {emb_str} |user {user_str}']

    for aid in row['slate']:
        feats = action_features_dict.get(int(aid), {})
        block_lines.append(
            f"|action aid_{int(aid)} "
            f"ch={feats.get('channel', 'NA')} "
            f"dw={feats.get('day', 'NA')} "
            f"tm={feats.get('time', 'NA')}"
        )

    vw_blocks.append('\n'.join(block_lines))
    if (idx + 1) % 10000 == 0:
        print(f'  ? Progress: {idx + 1:,} / {len(scoring_customers):,}')

scoring_path = Path('/tmp/vw_scoring_daily.txt')
SEP = chr(10)
scoring_path.write_text((SEP + SEP).join(vw_blocks) + SEP, encoding='utf-8')
print(f'Scoring file written: {scoring_path} ({scoring_path.stat().st_size / 1e6:.1f} MB)')
print(f'Scoring blocks built: {len(vw_blocks):,}')


In [ ]:
# ?? 5. Load Latest Model and Run VW Scoring ???????????????????????????????

print('Downloading latest trained model...')
model_key = f'{run_prefix_key}/vw_model_{USE_CASE_ID}.vw'
model_path = Path(f'/tmp/vw_model_{USE_CASE_ID}.vw')
download_s3(run_bucket, model_key, model_path)
print(f'Model downloaded from s3://{run_bucket}/{model_key}')

vw_bin = find_vw()
pred_path = Path('/tmp/vw_daily_predictions.txt')
cmd = [
    vw_bin,
    '-t',
    '-i', str(model_path),
    '-d', str(scoring_path),
    '-p', str(pred_path),
    '--quiet',
]
print(f'Running VW scoring for {len(vw_blocks):,} customers...')
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    raise RuntimeError(f'VW scoring failed:\n{proc.stderr}')

raw_preds = [line.strip() for line in pred_path.read_text().splitlines() if line.strip()]
if len(raw_preds) != len(scoring_customers):
    raise ValueError(
        f'Prediction count mismatch: predictions={len(raw_preds):,}, customers={len(scoring_customers):,}'
    )
raw_pred_values = [int(float(p.split()[0].split(',', 1)[0].split(':', 1)[0])) for p in raw_preds]
if raw_pred_values and min(raw_pred_values) >= 1 and max(raw_pred_values) <= SLATE_SIZE:
    predicted_action_indices = [p - 1 for p in raw_pred_values]
    pred_indexing = '1-based'
elif raw_pred_values and min(raw_pred_values) >= 0 and max(raw_pred_values) < SLATE_SIZE:
    predicted_action_indices = raw_pred_values
    pred_indexing = '0-based'
else:
    raise ValueError(f'Unexpected VW prediction range: {raw_pred_values[:10]}')

print(f'Predictions received: {len(predicted_action_indices):,}')
print(f'Detected indexing: {pred_indexing}')
print(f'Sample predictions: {predicted_action_indices[:10]}')


In [ ]:
# ?? 6. Build Output Recommendations ??????????????????????????????????????

print('Building output dataframe...')

decision_timestamp = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
rng = np.random.default_rng()
CHANNEL_CODE_TO_NAME = {
    'CH001': 'whatsapp',
    'CH002': 'sms',
    'CH003': 'rcs',
    'CH004': 'push',
    'CH005': 'email',
}

def normalize_channel(value) -> str:
    raw = str(value)
    return CHANNEL_CODE_TO_NAME.get(raw.upper(), raw.lower())

output_rows = []
explore_count = 0
for idx, pred_idx in enumerate(predicted_action_indices):
    if not (0 <= pred_idx < SLATE_SIZE):
        raise ValueError(f'Predicted index out of range at row {idx}: {pred_idx}')

    row = scoring_customers.iloc[idx]
    slate = [int(aid) for aid in row['slate']]
    action_probs = np.full(SLATE_SIZE, EXPLORATION_EPSILON / SLATE_SIZE, dtype=float)
    action_probs[pred_idx] += 1.0 - EXPLORATION_EPSILON
    chosen_idx = int(rng.choice(SLATE_SIZE, p=action_probs))
    if chosen_idx != pred_idx:
        explore_count += 1

    raw_master_user_id = row['master_user_id']
    master_user_id = str(raw_master_user_id)
    decision_id = f"{latest_run['run_id']}::{scoring_run_id}::{master_user_id}::{idx}"
    campaign_id = f"{scoring_run_id}_{idx:08d}"
    greedy_action_id = int(slate[pred_idx])
    recommended_action_id = int(slate[chosen_idx])
    action_details = action_features_dict.get(recommended_action_id, {})
    customer_segment = demo_lookup.get(raw_master_user_id, demo_lookup.get(master_user_id, {}))

    output_rows.append({
        'decision_id': decision_id,
        'use_case_id': USE_CASE_ID,
        'decision_ts': decision_timestamp,
        'customer_id': master_user_id,
        'master_user_id': master_user_id,
        'campaign_id': campaign_id,
        'model_run_id': latest_run['run_id'],
        'scoring_run_id': scoring_run_id,
        'decision_policy': DECISION_POLICY,
        'deployment_mode': deployment_mode,
        'latest_eval_production_promotable': bool(production_promotable),
        'latest_eval_scorecard': latest_eval.get('s3_scorecard'),
        'exploration_epsilon': float(EXPLORATION_EPSILON),
        'greedy_action_index': int(pred_idx),
        'greedy_action_id': greedy_action_id,
        'greedy_action_prob': float(action_probs[pred_idx]),
        'chosen_action_index': chosen_idx,
        'chosen_action_prob': float(action_probs[chosen_idx]),
        'recommended_action_id': recommended_action_id,
        'action_id': recommended_action_id,
        'channel_id': normalize_channel(action_details.get('channel', 'NA')),
        'channel': action_details.get('channel', 'NA'),
        'day': action_details.get('day', 'NA'),
        'time': action_details.get('time', 'NA'),
        'offer': action_details.get('offer', 'NA'),
        'income_tier': customer_segment.get('income_tier', 'NA'),
        'city_tier': customer_segment.get('city_tier', 'NA'),
        'lifecycle_stage': customer_segment.get('lifecycle_stage', 'NA'),
        'risk_level': customer_segment.get('risk_level', 'NA'),
        'seed_channel': row['seed_channel'],
        'seed_day': row['seed_day'],
        'seed_time': row['seed_time'],
        'slate_size': SLATE_SIZE,
        'candidate_slate_json': json.dumps(slate),
    })

decision_log_df = pd.DataFrame(output_rows)

required_decision_cols = [
    'decision_id', 'use_case_id', 'decision_ts', 'customer_id', 'master_user_id', 'campaign_id',
    'model_run_id', 'scoring_run_id',
    'decision_policy', 'deployment_mode', 'latest_eval_production_promotable',
    'exploration_epsilon', 'candidate_slate_json',
    'greedy_action_index', 'greedy_action_id', 'greedy_action_prob',
    'chosen_action_index', 'chosen_action_prob', 'recommended_action_id', 'action_id',
    'slate_size', 'channel_id'
]
missing_decision_cols = [c for c in required_decision_cols if c not in decision_log_df.columns]
if missing_decision_cols:
    raise ValueError(f'Decision log missing required columns: {missing_decision_cols}')
invalid_prob = ~decision_log_df['chosen_action_prob'].astype(float).between(0, 1, inclusive='right')
if invalid_prob.any():
    raise ValueError(f'Invalid chosen_action_prob rows: {int(invalid_prob.sum()):,}')
if decision_log_df['decision_id'].astype(str).duplicated().any():
    raise ValueError('Decision log has duplicate decision_id values.')
if decision_log_df['campaign_id'].astype(str).duplicated().any():
    raise ValueError('Decision log has duplicate campaign_id values.')
if decision_log_df[['master_user_id', 'campaign_id', 'action_id', 'channel_id']].isna().any().any():
    raise ValueError('Decision log has null production contract fields.')
missing_slate = decision_log_df['candidate_slate_json'].isna()
if missing_slate.any():
    raise ValueError(f'Missing candidate_slate_json rows: {int(missing_slate.sum()):,}')

def _slate_contains(row):
    slate = json.loads(row['candidate_slate_json'])
    return int(row['recommended_action_id']) in {int(aid) for aid in slate} and int(row['action_id']) == int(row['recommended_action_id'])
not_in_slate = ~decision_log_df.apply(_slate_contains, axis=1)
if not_in_slate.any():
    raise ValueError(f'Recommended action missing from slate rows: {int(not_in_slate.sum()):,}')
print('Decision log validation passed.')

recommendations_df = decision_log_df[[
    'customer_id',
    'master_user_id',
    'campaign_id',
    'model_run_id',
    'scoring_run_id',
    'recommended_action_id',
    'action_id',
    'channel',
    'channel_id',
    'day',
    'time',
    'offer',
    'income_tier',
    'city_tier',
    'lifecycle_stage',
    'risk_level',
    'deployment_mode',
]].copy()
print(f'Decision log shape   : {decision_log_df.shape}')
print(f'Recommendation shape : {recommendations_df.shape}')
print(f'Exploration assignments: {explore_count:,} ({explore_count / max(len(decision_log_df), 1):.1%})')
print(recommendations_df.head(10).to_string())

print('\n' + '=' * 70)
print('DAILY SCORING SUMMARY')
print('=' * 70)
for col, title in [
    ('channel', '[TOP CHANNELS RECOMMENDED TODAY]'),
    ('time', '[RECOMMENDED TIMINGS]'),
    ('day', '[RECOMMENDED DAYS]'),
]:
    print(f'\n{title}')
    for value, count in recommendations_df[col].value_counts().head(10).items():
        pct = count / len(recommendations_df) * 100
        print(f'  {value:<15}: {count:>8,} customers ({pct:>5.1f}%)')


In [ ]:
# ?? 7. Save Output & Upload to S3 ?????????????????????????????????????????

output_dir = Path('task8_artifacts') / deployment_mode / latest_run['run_id'] / scoring_run_id
output_dir.mkdir(parents=True, exist_ok=True)

date_tag = datetime.now(timezone.utc).strftime('%Y%m%d')
csv_path = output_dir / f'daily_recommendations_{date_tag}.csv'
parquet_path = output_dir / f'daily_recommendations_{date_tag}.parquet'
decision_log_csv_path = output_dir / f'decision_log_{date_tag}.csv'
decision_log_parquet_path = output_dir / f'decision_log_{date_tag}.parquet'
recommendations_df.to_csv(csv_path, index=False)
recommendations_df.to_parquet(parquet_path, index=False)
decision_log_df.to_csv(decision_log_csv_path, index=False)
decision_log_df.to_parquet(decision_log_parquet_path, index=False)

today = datetime.now(timezone.utc).strftime('%Y%m%d')
output_root = 'daily_recommendations' if deployment_mode == 'production' else 'shadow_recommendations'
prefix = f'{output_root}/{USE_CASE_ID}/{today}/{latest_run["run_id"]}/{scoring_run_id}'
latest_scoring_key = (
    f'model_artifacts/{USE_CASE_ID}/latest_scoring.json'
    if deployment_mode == 'production'
    else f'model_artifacts/{USE_CASE_ID}/latest_shadow_scoring.json'
)
print(f'Uploading to s3://{S3_CONFIG_BUCKET}/{prefix}/')
upload_s3(S3_CONFIG_BUCKET, f'{prefix}/recommendations.csv', csv_path)
upload_s3(S3_CONFIG_BUCKET, f'{prefix}/recommendations.parquet', parquet_path)
upload_s3(S3_CONFIG_BUCKET, f'{prefix}/decision_log.csv', decision_log_csv_path)
upload_s3(S3_CONFIG_BUCKET, f'{prefix}/decision_log.parquet', decision_log_parquet_path)

latest_decision_log_key = None
decision_log_prefix = None
decision_log_prod_csv_key = None
decision_log_prod_parquet_key = None
if deployment_mode == 'production':
    decision_log_prefix = f'decision_logs/{USE_CASE_ID}/production/{latest_run["run_id"]}/{scoring_run_id}'
    latest_decision_log_key = f'decision_logs/{USE_CASE_ID}/latest_production_decision_log.json'
    latest_decision_log_note = 'Production epsilon-greedy decision log. Use as Task 3/Task 4 input after outcomes are available.'
else:
    decision_log_prefix = f'decision_logs/{USE_CASE_ID}/shadow/{latest_run["run_id"]}/{scoring_run_id}'
    latest_decision_log_key = f'decision_logs/{USE_CASE_ID}/latest_shadow_decision_log.json'
    latest_decision_log_note = 'Shadow epsilon-greedy decision log. Use as Task 3/Task 4 input after outcomes are available.'

decision_log_prod_csv_key = f'{decision_log_prefix}/decision_log.csv'
decision_log_prod_parquet_key = f'{decision_log_prefix}/decision_log.parquet'
upload_s3(S3_CONFIG_BUCKET, decision_log_prod_csv_key, decision_log_csv_path)
upload_s3(S3_CONFIG_BUCKET, decision_log_prod_parquet_key, decision_log_parquet_path)
latest_decision_log = {
    'use_case_id': USE_CASE_ID,
    'run_id': latest_run['run_id'],
    'scoring_run_id': scoring_run_id,
    'decision_policy': DECISION_POLICY,
    'synthetic_replay_mode': False,
    'deployment_mode': deployment_mode,
    'generated_at_utc': datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    'decision_ts': decision_timestamp,
    's3_decision_log_parquet': f's3://{S3_CONFIG_BUCKET}/{decision_log_prod_parquet_key}',
    's3_decision_log_csv': f's3://{S3_CONFIG_BUCKET}/{decision_log_prod_csv_key}',
    'rows': int(len(decision_log_df)),
    'slate_size': int(SLATE_SIZE),
    'exploration_epsilon': float(EXPLORATION_EPSILON),
    'note': latest_decision_log_note,
}
s3_client.put_object(
    Bucket=S3_CONFIG_BUCKET,
    Key=latest_decision_log_key,
    Body=json.dumps(latest_decision_log, indent=2, default=str).encode('utf-8'),
    ContentType='application/json',
)

latest_scoring = {
    'run_id': latest_run['run_id'],
    'scoring_run_id': scoring_run_id,
    'use_case_id': USE_CASE_ID,
    'generated_at_utc': datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    'decision_policy': DECISION_POLICY,
    'deployment_mode': deployment_mode,
    'production_promotable': bool(production_promotable),
    'latest_eval_scorecard': latest_eval.get('s3_scorecard'),
    'production_blockers': latest_eval.get('production_blockers', []),
    'exploration_epsilon': float(EXPLORATION_EPSILON),
    's3_recommendations_csv': f's3://{S3_CONFIG_BUCKET}/{prefix}/recommendations.csv',
    's3_recommendations_parquet': f's3://{S3_CONFIG_BUCKET}/{prefix}/recommendations.parquet',
    's3_decision_log_csv': f's3://{S3_CONFIG_BUCKET}/{prefix}/decision_log.csv',
    's3_decision_log_parquet': f's3://{S3_CONFIG_BUCKET}/{prefix}/decision_log.parquet',
    'latest_decision_log_pointer': f's3://{S3_CONFIG_BUCKET}/{latest_decision_log_key}' if latest_decision_log_key else None,
    'customers_scored': int(len(recommendations_df)),
}
s3_client.put_object(
    Bucket=S3_CONFIG_BUCKET,
    Key=latest_scoring_key,
    Body=json.dumps(latest_scoring, indent=2).encode('utf-8'),
)

print('\n' + '=' * 70)
print('DAILY SCORING COMPLETE')
print('=' * 70)
print(f'Total customers scored: {len(recommendations_df):,}')
print(f'Run ID               : {latest_run["run_id"]}')
print(f'Scoring run ID       : {scoring_run_id}')
print(f'Decision policy      : {DECISION_POLICY} (epsilon={EXPLORATION_EPSILON:.2f})')
print(f'Deployment mode      : {deployment_mode}')
print(f'Production promotable: {production_promotable}')
print(f'Output CSV           : s3://{S3_CONFIG_BUCKET}/{prefix}/recommendations.csv')
print(f'Output Parquet       : s3://{S3_CONFIG_BUCKET}/{prefix}/recommendations.parquet')
print(f'Decision Log CSV     : s3://{S3_CONFIG_BUCKET}/{prefix}/decision_log.csv')
print(f'Decision Log Parquet : s3://{S3_CONFIG_BUCKET}/{prefix}/decision_log.parquet')
print(f'Latest scoring ptr   : s3://{S3_CONFIG_BUCKET}/{latest_scoring_key}')
if latest_decision_log_key:
    print(f'Latest decision ptr  : s3://{S3_CONFIG_BUCKET}/{latest_decision_log_key}')
